# UC-02 · Verify pump performance (FAT) - P-9001A in native units

**Use case:** [UC-02](../docs/product/use-cases.md) - *rated nameplate + measured
test points -> fitted H-Q / P-Q / eta-Q curves, deviation vs rated, pass / fail
verdict.*

The data is the Vendor A test sheet of pump **P-9001A**, entered **exactly as
printed**: flow in gpm, differential pressure in psi, water temperature in degF, motor
output in hp and the pump speed in rpm.  There is no hand conversion: Pint carries the
units and the library normalises them.

Steps: head and efficiency per row (with the water density at each row's
temperature) -> pin them on `TestPoint`s -> `to_speed` (10 301 -> 10 212 rpm) ->
`to_fluid` (test water -> rated fluid, SG 0.999) -> degree-3 fit -> `PerformanceChecker`
against **API 610 8th edition + client overlay N-553**.

> The API 610 8th-edition tolerances (Table 4-2) and the N-553 rev E overlay were
> checked against the standard texts; see
> [acceptance tolerances](../docs/concepts/acceptance-tolerances.md).  Efficiency is
> **not a rated value** in API 610, so it is reported for information only.

**Acceptance:** at the rated flow 9.80 m3/h the predicted head is about 339.5 m
(+1.7 % vs 333.8 m, inside +/-2 %), power about 24.6 kW and efficiency about 36.8 %.

## 1 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from pump import (
    AcceptanceCriteria,
    DesignPoint,
    Fluid,
    PerformanceChecker,
    PerformanceCurve,
    Q_,
    TestPoint,
    Water,
    deviation,
)
from pump.constants import G

## 2 · The test sheet, as printed

One row per test point: flow [gpm], differential pressure [psi], water temperature
[degF], motor output [hp].  The pump speed is a single reading.

In [ ]:
sheet = [
    # gpm,  dP psi,  T degF,  motor hp
    (15.2, 524.7, 130, 25.41),
    (29.4, 522.1, 129, 28.83),
    (34.8, 513.4, 129, 30.56),
    (43.5, 486.5, 127, 33.36),
    (47.9, 459.5, 126, 35.09),
    (52.2, 431.7, 126, 36.61),
]
pump_speed = Q_(10301, "rpm")

rated = dict(
    q=Q_(9.80, "m**3/h"),
    head=Q_(333.8, "m"),
    speed=Q_(10212, "rpm"),
    power=Q_(26.9, "kW"),
    efficiency=Q_(32.7, "percent"),
)
rated_fluid = Fluid("Wash water (rated)", density=Q_(999.0, "kg/m**3"))  # SG 0.999
print("rated:", {k: f"{v:~P}" for k, v in rated.items()})

## 3 · Head and efficiency per row

Each row is evaluated with **its own** water density (temperature dependent):
`TestPoint.head` is the pressure head `dP / (rho g)` and `TestPoint.efficiency` is the
hydraulic power over the breaking (motor output) power.

In [ ]:
rows = []
print(f"{'Q gpm':>6} {'Q m3/h':>7} {'rho kg/m3':>9} {'Head m':>8} {'P kW':>7} {'eta %':>6}")
for gpm, dp, t_f, hp in sheet:
    rho = Water(Q_(t_f, "degF")).density
    tp = TestPoint(
        fluid=Fluid("row water", density=rho),
        capacity=Q_(gpm, "gallon/min"),
        delta_pressure=Q_(dp, "psi"),
        breaking_power=Q_(hp, "hp"),
    )
    rows.append((rho, tp))
    print(
        f"{gpm:6.1f} {tp.capacity.magnitude:7.2f} {rho.magnitude:9.1f} "
        f"{tp.head.to('m').magnitude:8.2f} {tp.breaking_power.to('kW').magnitude:7.2f} "
        f"{tp.efficiency.to('percent').magnitude:6.1f}"
    )

# hand check of the first row with Pint: head = dP / (rho g)
rho0, tp0 = rows[0]
hand = (Q_(sheet[0][1], "psi") / (rho0 * G)).to("m")
print(f"\nrow 1 by hand: {hand:.3f~P}  | library: {tp0.head.to('m'):.3f~P}")

### The complete head (pressure + velocity + elevation)

When the bench geometry is known, `TestPoint` adds the velocity and elevation heads.
Here row 3 gets illustrative pipe IDs and gauge elevations: the total head is the sum
of the three components.

In [ ]:
rho3, tp3 = rows[2]
geo = TestPoint(
    fluid=Fluid("row water", density=rho3),
    capacity=tp3.capacity,
    delta_pressure=Q_(sheet[2][1], "psi"),
    inlet_diameter=Q_(2.0, "inch"),
    outlet_diameter=Q_(1.0, "inch"),
    inlet_elevation=Q_(0.3, "m"),
    outlet_elevation=Q_(0.8, "m"),
)
parts = {
    "pressure": geo.pressure_head.to("m"),
    "velocity": geo.velocity_head.to("m"),
    "elevation": geo.elevation_head.to("m"),
}
for name, value in parts.items():
    print(f"{name:>10}: {value:.4f~P}")
print(f"{'total':>10}: {geo.compute_head.to('m'):.4f~P}")

## 4 · Speed and density correction, then the fit

The measured points are pinned (`_head`, `_efficiency`) on a curve whose single fluid
is the mean test water (just a label, since Head and eta are pinned).  Then the
curve is corrected to the rated speed with the affinity laws and to the rated fluid.
Both transforms return **new** curves.

In [ ]:
rho_mean = Q_(float(np.mean([r.to("kg/m**3").magnitude for r, _ in rows])), "kg/m**3")
test_water = Fluid("Test water (mean)", density=rho_mean)

pinned = [
    TestPoint(
        fluid=test_water,
        capacity=tp.capacity,
        speed_of_rotation=pump_speed,
        _head=tp.head,
        _efficiency=tp.efficiency,
        breaking_power=tp.breaking_power,
    )
    for _, tp in rows
]
measured = PerformanceCurve(test_water, pinned, polynomial_degree=3)
speed_corrected = measured.to_speed(rated["speed"])
corrected = speed_corrected.to_fluid(rated_fluid)

print(f"{'Q_corr m3/h':>12} {'H_corr m':>9} {'P_corr kW':>10} {'eta %':>6}")
for p in corrected.points:
    print(
        f"{p.capacity.to('m**3/h').magnitude:12.3f} {p.head.to('m').magnitude:9.2f} "
        f"{p.breaking_power.to('kW').magnitude:10.2f} {p.efficiency.to('percent').magnitude:6.2f}"
    )

## 5 · Check against the rated point

`AcceptanceCriteria.load("api610:8", client="n553")` selects the edition and the client
overlay.  API 610 8th ed. Table 4-2: the rated head 333.8 m is above 300 m, so the head
band is -2/+2 %, the shut-off band +/-5 % and the power limit +4 %.  The N-553 rev E
overlay adds the NPSH margin rule (>= 0.8 m and NPSHa/NPSHr >= 1.15 at rated flow).

In [ ]:
criteria = AcceptanceCriteria.load("api610:8", client="n553")
design = DesignPoint(
    fluid=rated_fluid,
    capacity=rated["q"],
    differential_head=rated["head"],
    breaking_power=rated["power"],
)
checker = PerformanceChecker(design, corrected, criteria=criteria)

print(criteria.label, "+", criteria.client_label, "| verified:", criteria.verified)
print("unverified parts:", ", ".join(criteria.unverified))
print()
print(checker.check_summary)
print()
print({k: f"{v:~P}" for k, v in checker.acceptable_limits.items()})

## 6 · The corrected curves

In [ ]:
png = corrected.plot_performance_curve(
    chart_title="P-9001A - corrected to 10212 rpm, SG 0.999",
    capacity=rated["q"],
    return_io=True,
)
display(Image(data=png.getvalue()))

## 7 · Acceptance

The numbers below are the project's regression fixture (sprint 1-3 changelog).

In [ ]:
pred_h = corrected.predict_head(rated["q"]).to("m")
pred_p = corrected.predict_breaking_power(rated["q"]).to("kW")
pred_e = corrected.predict_efficiency(rated["q"]).to("percent")
dev_h = deviation(rated["head"], pred_h)
print(f"H = {pred_h:.2f~P}  P = {pred_p:.2f~P}  eta = {pred_e:.2f~P}  dH = {dev_h:+.2%}")

# the head per row is dP/(rho g): the library equals the hand computation
assert abs((hand - tp0.head).to("m").magnitude) < 1e-9

# the complete head is the sum of its three components
total = parts["pressure"] + parts["velocity"] + parts["elevation"]
assert abs((total - geo.compute_head).to("m").magnitude) < 1e-9
assert parts["velocity"].magnitude > 0 and abs(parts["elevation"].magnitude - 0.5) < 1e-9

# affinity: capacity scales with the speed ratio
ratio = 10212 / 10301
assert abs(corrected.points[0].capacity.magnitude - measured.points[0].capacity.magnitude * ratio) < 1e-9

# predicted values at the rated flow (regression fixture)
assert abs(pred_h.magnitude - 339.55) < 0.6
assert abs(pred_p.magnitude - 24.60) < 0.35
assert abs(pred_e.magnitude - 36.80) < 0.5
assert 0.015 < dev_h < 0.019

# verdict: inside +/-2 % head, power below +4 %
assert checker.minimum_head <= pred_h <= checker.maximum_head
assert pred_p <= checker.maximum_breaking_power
assert "False" not in checker.check_summary

# the performance tolerances are checked against API 610 8th ed. Table 4-2;
# only the mechanical-run limits remain open (no run test in the 8th edition)
assert criteria.verified is True
assert criteria.performance_unverified == ()
assert criteria.efficiency_tol_minus is None  # "Efficiency is not a rating value"
print("UC-02 acceptance OK")